# E10 - TĂNG CƯỜNG DỮ LIỆU (AUGMENTATION) CHO LỚP TRUNG LẬP (NGƯỜI 4)

**Vấn đề:** Lớp Trung lập ít mẫu nhất (`1.524` câu train) và khó học.
**Thiết kế:** Chỉ tăng cường trên lớp `sentiment == 1` **bên trong tập train**, giữ nguyên `dev_clean` để tránh rò rỉ. Thử 3 kỹ thuật phù hợp văn bản tiếng Việt tách từ (`Random Swap`, `Random Deletion 10%`, `Reverse Teencode`) với hệ số nhân `x1` (+50%) và `x2` (+100%).

In [1]:
import os, sys, glob, zipfile, subprocess

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

def _find_root():
    cwd = os.getcwd()
    cand = os.path.abspath('..') if os.path.basename(cwd) in ('notebooks', 'experiments') else cwd
    if os.path.exists(os.path.join(cand, 'src', 'modeling.py')):
        return cand, cand
    if IN_COLAB:
        for base in [cwd, '/content']:
            if os.path.exists(os.path.join(base, 'src', 'modeling.py')):
                return base, base
        for z in glob.glob('/content/*src_and_data*.zip'):
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall('/content')
            if os.path.exists('/content/src/modeling.py'):
                return '/content', '/content'
    if IN_KAGGLE:
        work = '/kaggle/working'
        if os.path.exists(os.path.join(work, 'src', 'modeling.py')):
            return work, work
        for p in glob.glob('/kaggle/input/**/src/modeling.py', recursive=True):
            return os.path.dirname(os.path.dirname(p)), work
        for z in glob.glob('/kaggle/input/**/*src_and_data*.zip', recursive=True):
            dest = os.path.join(work, 'project')
            os.makedirs(dest, exist_ok=True)
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall(dest)
            if os.path.exists(os.path.join(dest, 'src', 'modeling.py')):
                return dest, dest
    raise FileNotFoundError('Khong tim thay src/modeling.py! Hay chay trong repo hoac upload src_and_data.zip.')

ROOT, WORK_DIR = _find_root()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
PROC_DIR = os.path.join(ROOT, 'data', 'processed')
FIG_DIR = os.path.join(WORK_DIR, 'reports', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
os.environ['EXP_LOG_PATH'] = os.path.join(WORK_DIR, 'reports', 'experiments_log.csv')
print(f'Env: colab={IN_COLAB}, kaggle={IN_KAGGLE} | ROOT={ROOT}')

Env: colab=False, kaggle=False | ROOT=D:\E-commerce


In [2]:
import random, re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score
from src.text_cleaner import _TEENCODE_DICT
from src.exp_log import log_experiment

df_train = pd.read_csv(os.path.join(PROC_DIR, 'train_clean.csv'))
df_dev = pd.read_csv(os.path.join(PROC_DIR, 'dev_clean.csv'))
X_dev, y_dev = df_dev['sentence_seg'].fillna(''), df_dev['sentiment'].values
TOKEN_PAT = r'(?u)\b\w+\b|\[\w+\]'

# Tao tu dien teencode nguoc an toan (tu chuan -> cac bien the teencode)
REV_TEENCODE = {}
for k, v in _TEENCODE_DICT.items():
    v_seg = v.replace(' ', '_')
    if v_seg not in ('ok',) and len(k) >= 2:
        REV_TEENCODE.setdefault(v_seg, []).append(k)
REV_TEENCODE['không'] = ['ko', 'hok', 'k']

PROTECT_NEG = {'không', 'chưa', 'chẳng', 'chả', 'kém', 'thiếu'}

def aug_swap(text, rng):
    toks = str(text).split()
    if len(toks) < 4:
        return text
    idx = rng.randrange(len(toks) - 1)
    if not toks[idx].startswith('[') and not toks[idx + 1].startswith('['):
        toks[idx], toks[idx + 1] = toks[idx + 1], toks[idx]
    return ' '.join(toks)

def aug_del(text, rng, p=0.10):
    toks = str(text).split()
    if len(toks) < 6:
        return text
    kept = [t for t in toks if t.startswith('[') or t.lower() in PROTECT_NEG or rng.random() > p]
    return ' '.join(kept) if len(kept) >= 3 else text

def aug_rev_teencode(text, rng):
    toks = str(text).split()
    out = []
    for t in toks:
        low = t.lower()
        if low in REV_TEENCODE and rng.random() < 0.5:
            out.append(rng.choice(REV_TEENCODE[low]))
        else:
            out.append(t)
    return ' '.join(out)

df_neu = df_train[df_train['sentiment'] == 1]
sample_s = df_neu['sentence_seg'].iloc[0]
print('Goc:', sample_s)
rng_demo = random.Random(42)
print('Swap:', aug_swap(sample_s, rng_demo))
print('Del :', aug_del(sample_s, rng_demo))
print('Teen:', aug_rev_teencode(sample_s, rng_demo))

Goc: Pin kém còn lại miễn chê mua 8 / 3 / 2019 tình_trạng pin còn 88 % có ai giống tôi không
Swap: Pin kém còn lại miễn chê mua 8 / 3 / 2019 tình_trạng pin còn 88 % có ai giống không tôi
Del : Pin kém còn lại miễn chê mua 8 / 2019 tình_trạng pin còn 88 % có ai giống tôi không
Teen: Pin kém còn lại miễn chê mua 8 / 3 / 2019 tình_trạng pin còn 88 % có ai giống tôi hok


In [3]:
def run_aug_eval(fn, mult, cw, name):
    rng = random.Random(42)
    if fn is None:
        df_aug = df_train
    else:
        n_add = int(len(df_neu) * mult)
        sampled = df_neu.sample(n=n_add, replace=(mult > 1), random_state=42).copy()
        sampled['sentence_seg'] = [fn(t, rng) for t in sampled['sentence_seg'].fillna('')]
        df_aug = pd.concat([df_train, sampled], ignore_index=True)

    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(token_pattern=TOKEN_PAT, ngram_range=(1, 2), min_df=2, max_df=0.95, max_features=15000, sublinear_tf=True)),
        ('clf', LogisticRegression(C=1.0, class_weight=cw, max_iter=1000, random_state=42)),
    ]).fit(df_aug['sentence_seg'].fillna(''), df_aug['sentiment'].values)
    pred = pipe.predict(X_dev)
    mf1 = f1_score(y_dev, pred, average='macro')
    neu_f1 = f1_score(y_dev, pred, labels=[1], average='macro')
    log_experiment('E10', name, 'Data augmentation on Neutral train only',
                   {'method': name, 'mult': mult, 'class_weight': cw, 'n_train': len(df_aug)},
                   dev_f1=mf1, notes=f'neu_f1={neu_f1:.4f}')
    return {'run': name, 'n_train': len(df_aug), 'cw': str(cw), 'dev_macro_f1': round(mf1, 4), 'neu_f1': round(neu_f1, 4)}

runs = [
    run_aug_eval(None, 0.0, 'balanced', 'baseline_no_aug_balanced'),
    run_aug_eval(None, 0.0, None, 'baseline_no_aug_none'),
]
for mname, fn in [('swap', aug_swap), ('del10', aug_del), ('rev_teen', aug_rev_teencode)]:
    for mult, tag in [(0.5, 'x0.5'), (1.0, 'x1.0')]:
        runs.append(run_aug_eval(fn, mult, 'balanced', f'{mname}_{tag}_bal'))
        runs.append(run_aug_eval(fn, mult, None, f'{mname}_{tag}_none'))

pd.DataFrame(runs).sort_values('dev_macro_f1', ascending=False)

,run,n_train,cw,dev_macro_f1,neu_f1
5,swap_x1.0_none,9300,None,0.7535,0.5495
9,del10_x1.0_none,9300,None,0.7527,0.5467
2,swap_x0.5_bal,8538,balanced,0.7515,0.5482
7,del10_x0.5_none,8538,None,0.7489,0.5248
8,del10_x1.0_bal,9300,balanced,0.7488,0.5478
6,del10_x0.5_bal,8538,balanced,0.7488,0.5447
3,swap_x0.5_none,8538,None,0.7481,0.5237
0,baseline_no_aug_balanced,7776,balanced,0.7478,0.5466
4,swap_x1.0_bal,9300,balanced,0.7469,0.5447
10,rev_teen_x0.5_bal,8538,balanced,0.7421,0.5275


## 2. KẾT LUẬN & HƯỚNG GIẢI QUYẾT ĐỀ XUẤT

- **Kết quả trên Dev (chỉ augment lớp Trung lập trong train):**
  - `swap_x1.0_none` (đảo nhẹ 1 cặp từ cạnh nhau cho +100% câu Trung lập, không cần `class_weight='balanced'`) đạt **`dev_macro_f1 = 0.7535`** (`neu_f1 = 0.5495`), cao hơn baseline `balanced` (`0.7478`) và tăng **`+0.0385`** so với không cân bằng (`0.7150`). `del10_x1.0_none` cũng đạt `0.7527`.
  - Ngược lại, **`rev_teencode` làm GIẢM điểm** (`0.7280–0.7421` < `0.7478`) vì tập `dev_clean` đã chuẩn hoá teencode rồi — sinh teencode ngược lại vào tập train tạo nhiễu từ vựng lệch phân bố với đầu vào sạch.
- **Hướng giải quyết (cho NB03):** nếu dùng TF-IDF không trọng số, có thể bật `Random Swap (x1.0)` chỉ trên fold train (hoặc dùng `oversample_neu` ở E09 cho đơn giản, tương đương `0.7521`); **tuyệt đối không** dùng `Reverse Teencode` sau khi văn bản đã qua bước `clean_text`.
